# Urban Flood — Model 2 / NodeType 1 dataset builder + CatBoost + BiGRU ensemble


In [1]:
from __future__ import annotations

import gc
import os
import re
import time
import copy
import random
import warnings
import ctypes
from collections import defaultdict
from pathlib import Path

from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh

import numpy as np
import pandas as pd
import pandas.api.types
from catboost import CatBoostRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

In [2]:
# =========================================================
# CONFIG
# =========================================================
DATA_ROOT = Path("/kaggle/input/datasets/bulbazavril/fullds/Models")

KEY_COLS = ["model_id", "event_id", "node_type", "node_id"]
TARGET_COL = "water_level"
SUBMISSION_COLS = ["row_id"] + KEY_COLS + [TARGET_COL]

VALID_EVENTS_PER_MODEL = 15
RANDOM_SEED = 42
MIN_PRED_TIMESTEP = 10
WARMUP_STEPS = 10
USE_TARGET_DELTA = True

USE_SPECTRAL_EMBED = False   # Laplacian eigenvector embeddings
SPECTRAL_DIM = 8             # Number of eigenvectors
USE_AUX_TARGETS = False      # Predict inlet_flow → use as feature

CB_VALID_SEED = 56
CB_REFIT_SEEDS = [56, 77, 99]


FINAL_TEST_PRED_PARQUET_PATH = Path("submission_m2_nt1_refit_cb_xgb_ensemble.parquet")
FINAL_TEST_COMPONENTS_PARQUET_PATH = Path("submission_m2_nt1_refit_cb_xgb_components.parquet")

print("Config ready: CatBoost + XGBoost ensemble with GPU boosting and parquet export.")

Config ready: CatBoost + XGBoost ensemble with GPU boosting and parquet export.


In [3]:
# =========================================================
# METRIC
# =========================================================
STD_DEV_DICT = {
    (1, 1): 16.877747, (1, 2): 14.378797,
    (2, 1): 3.191784,  (2, 2): 2.727131,
}

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def local_score(solution_df, submission_df):
    sol = solution_df.copy()
    sub = submission_df.copy()
    model_scores = []
    for mid in sorted(sol['model_id'].unique()):
        nt_raw = {1: [], 2: []}
        event_scores = []
        for evid in sorted(sol[sol['model_id']==mid]['event_id'].unique()):
            nt_scores = []
            for nt in [1, 2]:
                mask_s = (sol['model_id']==mid) & (sol['event_id']==evid) & (sol['node_type']==nt)
                mask_b = (sub['model_id']==mid) & (sub['event_id']==evid) & (sub['node_type']==nt)
                if mask_s.sum() == 0: continue
                s_df = sol[mask_s]; b_df = sub[mask_b]
                sd = STD_DEV_DICT.get((mid, nt), 1.0)
                node_std_rmses = []
                for nid in s_df['node_id'].unique():
                    nm_s = s_df[s_df['node_id']==nid]['water_level'].values
                    nm_b = b_df[b_df['node_id']==nid]['water_level'].values
                    if len(nm_s) > 1 and len(nm_s) == len(nm_b):
                        r = rmse(nm_s, nm_b)
                        node_std_rmses.append(r / sd)
                        nt_raw[nt].append(r)
                if node_std_rmses:
                    nt_scores.append(np.mean(node_std_rmses))
            if nt_scores:
                event_scores.append(np.mean(nt_scores))
        if event_scores:
            ms = np.mean(event_scores)
            model_scores.append(ms)
            print(f"  Model {mid}: Std RMSE = {ms:.6f} ({len(event_scores)} events)")
            for nt in [1, 2]:
                if nt_raw[nt]:
                    print(f"    NT{nt}: raw RMSE mean={np.mean(nt_raw[nt]):.6f}")
    final = np.mean(model_scores)
    print(f"  Overall: {final:.6f}")
    return final

In [4]:
# =========================================================
# UTILITIES
# =========================================================
def _free_memory():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except:
        pass

def ncol(col):
    col = str(col).strip().lower()
    col = col.replace('\u00b2', '2').replace('\u00b3', '3').replace('%', 'pct')
    col = re.sub(r'[^a-z0-9]+', '_', col)
    return re.sub(r'_+', '_', col).strip('_')

def read_csv(path):
    df = pd.read_csv(path)
    df.columns = [ncol(c) for c in df.columns]
    drop = [c for c in df.columns if c.startswith('unnamed')]
    if drop: df = df.drop(columns=drop)
    return df

def read_table(path):
    if str(path).endswith('.parquet'): df = pd.read_parquet(path)
    else: df = pd.read_csv(path)
    df.columns = [ncol(c) for c in df.columns]
    return df

def fcol(columns, names=(), tokens=(), required=True, label='col'):
    cols = list(columns)
    for n in names:
        if n in cols: return n
    for c in cols:
        if all(t in c for t in tokens): return c
    if required: raise KeyError(f"Cannot find {label} in {cols}")
    return None

def ensure_nid(df):
    c = fcol(df.columns, ('node_idx','node_id','id'), ('node',), False, 'nid')
    out = df.copy()
    if c is None: out['node_id'] = np.arange(len(out), dtype='int64')
    elif c != 'node_id': out = out.rename(columns={c: 'node_id'})
    out['node_id'] = out['node_id'].astype('int64')
    return out

def eid(d): return int(d.name.split('_')[1])
def mid_fn(d): return int(d.name.split('_')[1])
def event_dirs(split_dir):
    return sorted([d for d in split_dir.glob('event_*') if d.is_dir()], key=eid)

def dyn_path(event_dir, nt):
    return event_dir / f"{'1d' if nt==1 else '2d'}_nodes_dynamic_all.csv"

def edge_dyn_path(event_dir, nt):
    return event_dir / f"{'1d' if nt==1 else '2d'}_edges_dynamic_all.csv"

def find_file(model_dir, filename):
    for s in ('train', 'test'):
        p = model_dir / s / filename
        if p.exists(): return p
    return None

def downcast_float32(df):
    for c in df.columns:
        if df[c].dtype == 'float64':
            df[c] = df[c].astype('float32')
    return df

print("Utilities ready.")

Utilities ready.


In [5]:
# =========================================================
# STATIC NODE FEATURES
# =========================================================
STATIC_CACHE = {}

def load_static(model_id, node_type):
    k = (model_id, node_type)
    if k in STATIC_CACHE: return STATIC_CACHE[k]
    md = DATA_ROOT / f'Model_{model_id}'
    if node_type == 1:
        cands = ['1d_nodes_static.csv']
    else:
        cands = ['2d_nodes_static.csv', '2d_nodes_index.csv']
    for fn in cands:
        p = find_file(md, fn)
        if p:
            df = read_csv(p)
            df = ensure_nid(df)
            df = df.drop_duplicates('node_id').reset_index(drop=True)
            STATIC_CACHE[k] = df
            return df
    raise FileNotFoundError(f'Static not found for m{model_id} nt{node_type}')

print("Static loader ready.")

Static loader ready.


In [6]:
# =========================================================
# GRAPH FEATURES + MULTI-HOP + CENTRALITY + PIPE CAPACITY
# =========================================================
GRAPH_CACHE = {}
ADJ_CACHE = {}
EDGE_INDEX_CACHE = {}

def _count_upstream_total(upstream_dict, all_nids):
    cache = {}
    def _get_ancestors(nid, visited):
        if nid in cache: return cache[nid]
        if nid in visited: return set()
        visited.add(nid)
        result = set()
        for parent in upstream_dict.get(nid, []):
            result.add(parent)
            result.update(_get_ancestors(parent, visited))
        cache[nid] = result
        return result
    out = {}
    for nid in all_nids:
        out[int(nid)] = len(_get_ancestors(int(nid), set()))
    return out

def _compute_2hop_features(adj, node_feat_map, all_nids, feat_name):
    """Compute 2-hop neighbor aggregation for a given feature."""
    result = {}
    for nid in all_nids:
        nid = int(nid)
        hop1 = adj.get(nid, [])
        hop2_set = set()
        for n1 in hop1:
            for n2 in adj.get(n1, []):
                if n2 != nid and n2 not in hop1:
                    hop2_set.add(n2)
        if hop2_set:
            vals = [node_feat_map.get(n, np.nan) for n in hop2_set]
            vals = [v for v in vals if not np.isnan(v)]
            if vals:
                result[nid] = {
                    f'hop2_{feat_name}_mean': np.mean(vals),
                    f'hop2_{feat_name}_max': np.max(vals),
                    f'hop2_{feat_name}_min': np.min(vals),
                    f'hop2_count': len(hop2_set),
                }
    return result

def _pipe_capacity_manning(diameter, roughness, slope):
    """Manning's formula for full-pipe capacity: Q = (1/n) * A * R^(2/3) * S^(1/2).
    Circular pipe: A = pi*d^2/4, R = d/4.
    """
    if diameter <= 0 or roughness <= 0 or slope <= 0:
        return 0.0
    area = np.pi * diameter**2 / 4.0
    hyd_radius = diameter / 4.0
    return (1.0 / roughness) * area * hyd_radius**(2.0/3.0) * np.sqrt(slope)

def load_graph_feats(model_id, node_type):
    k = (model_id, node_type)
    if k in GRAPH_CACHE: return GRAPH_CACHE[k]
    md = DATA_ROOT / f'Model_{model_id}'
    prefix = '1d' if node_type == 1 else '2d'
    ei_path = find_file(md, f'{prefix}_edge_index.csv')

    if ei_path is None:
        print(f"  [WARN] {prefix}_edge_index.csv not found for Model_{model_id}")
        GRAPH_CACHE[k] = pd.DataFrame()
        return GRAPH_CACHE[k]

    ei = read_csv(ei_path)
    print(f"  Model {model_id} {prefix} edge_index: {len(ei)} edges, cols={list(ei.columns)}")

    cols = list(ei.columns)
    fc = fcol(cols, ('from_node','from_node_id','source'), ('from',), False, 'from')
    tc = fcol(cols, ('to_node','to_node_id','target'), ('to',), False, 'to')
    if fc is None or tc is None: fc, tc = cols[0], cols[1]

    EDGE_INDEX_CACHE[k] = (fc, tc, ei)

    adj = defaultdict(set)
    downstream = defaultdict(list)
    upstream = defaultdict(list)
    for _, r in ei.iterrows():
        u, v = int(r[fc]), int(r[tc])
        adj[u].add(v); adj[v].add(u)
        downstream[u].append(v)
        upstream[v].append(u)
    adj = {k_: list(v) for k_, v in adj.items()}
    ADJ_CACHE[k] = adj

    # Load edge static features
    es_path = find_file(md, f'{prefix}_edges_static.csv')
    edge_static = read_csv(es_path) if es_path else None
    if edge_static is not None:
        print(f"    Edge static: {len(edge_static)} edges, cols={list(edge_static.columns)}")

    efmap = {}
    if edge_static is not None and len(ei) == len(edge_static):
        for i in range(len(ei)):
            u, v = int(ei.iloc[i][fc]), int(ei.iloc[i][tc])
            efmap[(u, v)] = edge_static.iloc[i].to_dict()

    # 1D-2D connections
    conn = find_file(md, '1d2d_connections.csv')
    conn_map = defaultdict(list)
    conn_1d_to_2d = defaultdict(list)
    if conn:
        cdf = read_csv(conn)
        cc = list(cdf.columns)
        c1 = fcol(cc, ('1d_node_id','node_1d'), ('1d',), False, '1d')
        c2 = fcol(cc, ('2d_node_id','node_2d'), ('2d',), False, '2d')
        if c1 is None or c2 is None: c1, c2 = cc[0], cc[1]
        for _, r in cdf.iterrows():
            n1, n2 = int(r[c1]), int(r[c2])
            conn_1d_to_2d[n1].append(n2)
            if node_type == 1: conn_map[n1].append(n2)
            else: conn_map[n2].append(n1)

    static = load_static(model_id, node_type)
    all_nids = static['node_id'].unique()

    # Transitive upstream count (1D only)
    upstream_total = {}
    if node_type == 1:
        upstream_total = _count_upstream_total(upstream, all_nids)
        print(f"    Upstream total: max={max(upstream_total.values()) if upstream_total else 0}")

    # Elevation maps for gradients + 2-hop
    elev_map = {}
    if node_type == 2:
        elev_c = fcol(static.columns, ('elevation','min_elevation','centroid_elevation'),
                      ('elev',), False, 'elev')
        if elev_c:
            elev_map = dict(zip(static['node_id'].astype(int), static[elev_c].astype(float)))
    elif node_type == 1:
        inv_c = fcol(static.columns, ('invert_elevation',), ('invert',), False, 'inv')
        if inv_c:
            elev_map = dict(zip(static['node_id'].astype(int), static[inv_c].astype(float)))

    # Enhanced coupling: elevation difference 1D<->2D
    coupling_elev_diff = {}
    if node_type == 1 and conn_1d_to_2d:
        s2d = load_static(model_id, 2)
        elev_c_2d = fcol(s2d.columns, ('elevation','min_elevation','centroid_elevation'),
                         ('elev',), False, 'elev')
        inv_c = fcol(static.columns, ('invert_elevation',), ('invert',), False, 'inv')
        if elev_c_2d and inv_c:
            elev_2d_map = dict(zip(s2d['node_id'].astype(int), s2d[elev_c_2d].astype(float)))
            inv_map = dict(zip(static['node_id'].astype(int), static[inv_c].astype(float)))
            for n1d, n2d_list in conn_1d_to_2d.items():
                e2d_vals = [elev_2d_map.get(n2, np.nan) for n2 in n2d_list]
                e2d_clean = [v for v in e2d_vals if not np.isnan(v)]
                inv_val = inv_map.get(n1d, np.nan)
                if e2d_clean and not np.isnan(inv_val):
                    coupling_elev_diff[n1d] = np.mean(e2d_clean) - inv_val

    # v5.5: Graph centrality via NetworkX (fast for small 1D graphs, ok for 2D)
    centrality_pr = {}
    centrality_deg = {}
    try:
        import networkx as nx
        G = nx.Graph()
        for _, r in ei.iterrows():
            G.add_edge(int(r[fc]), int(r[tc]))
        centrality_pr = nx.pagerank(G, max_iter=100, tol=1e-4)
        centrality_deg = nx.degree_centrality(G)
        print(f"    Centrality: PageRank + degree for {G.number_of_nodes()} nodes")
    except Exception as e:
        print(f"    [WARN] centrality failed: {e}")

    # v5.5: 2-hop features
    hop2_feats = {}
    if elev_map:
        hop2_feats = _compute_2hop_features(adj, elev_map, all_nids, 'elev')
        print(f"    2-hop elev features: {len(hop2_feats)} nodes")

    # v5.5: Pipe capacity (1D only, Manning's formula)
    pipe_cap_map = {}
    if node_type == 1 and efmap:
        node_cap_in = defaultdict(list)
        node_cap_out = defaultdict(list)
        for (u, v), ef in efmap.items():
            diam = float(ef.get('diameter', 0) or 0)
            rough = float(ef.get('roughness', 0) or 0)
            slp = abs(float(ef.get('slope', 0) or 0))
            cap = _pipe_capacity_manning(diam, rough, slp)
            node_cap_out[u].append(cap)
            node_cap_in[v].append(cap)
        for nid in all_nids:
            nid = int(nid)
            caps_in = node_cap_in.get(nid, [])
            caps_out = node_cap_out.get(nid, [])
            all_caps = caps_in + caps_out
            if all_caps:
                pipe_cap_map[nid] = {
                    'pipe_cap_total_in': sum(caps_in),
                    'pipe_cap_total_out': sum(caps_out),
                    'pipe_cap_max': max(all_caps),
                    'pipe_cap_mean': np.mean(all_caps),
                }
        print(f"    Pipe capacity: {len(pipe_cap_map)} nodes")

    recs = []
    for nid in all_nids:
        nid = int(nid)
        r = {'node_id': nid, 'degree': len(adj.get(nid, []))}

        # Centrality
        r['pagerank'] = centrality_pr.get(nid, 0.0)
        r['degree_centrality'] = centrality_deg.get(nid, 0.0)

        # 2-hop features
        if nid in hop2_feats:
            r.update(hop2_feats[nid])

        if node_type == 1:
            up = upstream.get(nid, [])
            dn = downstream.get(nid, [])
            r['n_upstream'] = len(up)
            r['n_downstream'] = len(dn)
            r['upstream_total'] = upstream_total.get(nid, 0)

            # Aggregate upstream/downstream edge static features
            up_feats = [efmap.get((u, nid), {}) for u in up]
            dn_feats = [efmap.get((nid, v), {}) for v in dn]
            for pname, flist in [('up', up_feats), ('dn', dn_feats)]:
                if flist:
                    feat_names = []
                    for fname in flist[0].keys():
                        try:
                            vals = [float(f.get(fname, np.nan)) for f in flist if pd.notna(f.get(fname, np.nan))]
                        except Exception:
                            vals = []
                        if vals:
                            feat_names.append(fname)
                    for fname in feat_names:
                        vals = []
                        for f in flist:
                            try:
                                v = float(f.get(fname, np.nan))
                                if pd.notna(v):
                                    vals.append(v)
                            except Exception:
                                pass
                        if vals:
                            arr = np.asarray(vals, dtype='float64')
                            r[f'{pname}_{fname}_mean'] = np.mean(arr)
                            r[f'{pname}_{fname}_min'] = np.min(arr)
                            r[f'{pname}_{fname}_max'] = np.max(arr)
                            r[f'{pname}_{fname}_std'] = np.std(arr) if arr.size > 1 else 0.0

            conn_nodes = conn_map.get(nid, [])
            r['has_2d_conn'] = int(len(conn_nodes) > 0)
            r['n_2d_conn'] = len(conn_nodes)
            r['coupling_elev_diff'] = coupling_elev_diff.get(nid, np.nan)

            # Pipe capacity
            if nid in pipe_cap_map:
                r.update(pipe_cap_map[nid])

        else:  # 2D
            conn_nodes = conn_map.get(nid, [])
            r['has_1d_conn'] = int(len(conn_nodes) > 0)
            r['n_1d_conn'] = len(conn_nodes)
            nbrs = adj.get(nid, [])
            if nbrs and elev_map:
                own_elev = elev_map.get(nid, np.nan)
                nbr_elevs = [elev_map.get(n, np.nan) for n in nbrs]
                nbr_elevs_clean = [e for e in nbr_elevs if not np.isnan(e)]
                if nbr_elevs_clean:
                    r['nbr_mean_elev'] = np.mean(nbr_elevs_clean)
                    r['nbr_min_elev'] = np.min(nbr_elevs_clean)
                    if not np.isnan(own_elev):
                        r['elev_gradient_mean'] = own_elev - np.mean(nbr_elevs_clean)
                        r['elev_gradient_max'] = own_elev - np.min(nbr_elevs_clean)

            # v5: aggregate 2D static edge features to nodes
            if efmap:
                nbr_edge_feats = []
                for nb in nbrs:
                    ef = efmap.get((nid, nb)) or efmap.get((nb, nid))
                    if ef: nbr_edge_feats.append(ef)
                if nbr_edge_feats:
                    for fname in list(nbr_edge_feats[0].keys())[:6]:
                        vals = []
                        for f in nbr_edge_feats:
                            try: vals.append(float(f.get(fname, 0)))
                            except: pass
                        if vals:
                            r[f'nbr_{fname}_mean'] = np.mean(vals)
        recs.append(r)

    result = pd.DataFrame(recs)
    result['node_id'] = result['node_id'].astype('int64')
    print(f"    Graph features: {len(result)} nodes, {len(result.columns)-1} features")
    GRAPH_CACHE[k] = result
    return result

# --- v6 Experimental: Spectral graph embeddings ---
SPECTRAL_CACHE = {}

def compute_spectral_embeddings(model_id, node_type):
    """Compute Laplacian eigenvector embeddings from graph structure."""
    if not USE_SPECTRAL_EMBED:
        return pd.DataFrame()
    k = (model_id, node_type)
    if k in SPECTRAL_CACHE:
        return SPECTRAL_CACHE[k]

    try:
        ei = load_edge_index(model_id, node_type)
        if ei is None or len(ei) == 0:
            SPECTRAL_CACHE[k] = pd.DataFrame()
            return pd.DataFrame()

        src_col = fcol(ei.columns, ('source','from','src'), ('node','id'), label='src')
        dst_col = fcol(ei.columns, ('target','to','dst','dest'), ('node','id'), label='dst')
        if src_col is None or dst_col is None:
            SPECTRAL_CACHE[k] = pd.DataFrame()
            return pd.DataFrame()

        nodes = sorted(set(ei[src_col].tolist() + ei[dst_col].tolist()))
        n = len(nodes)
        if n < SPECTRAL_DIM + 2:
            SPECTRAL_CACHE[k] = pd.DataFrame()
            return pd.DataFrame()

        nid_map = {v: i for i, v in enumerate(nodes)}
        rows = [nid_map[v] for v in ei[src_col]]
        cols = [nid_map[v] for v in ei[dst_col]]
        # Symmetric adjacency
        rows_full = rows + cols
        cols_full = cols + rows
        vals = np.ones(len(rows_full), dtype='float32')
        A = csr_matrix((vals, (rows_full, cols_full)), shape=(n, n))
        # Remove self-loops, ensure binary
        A.setdiag(0)
        A.eliminate_zeros()
        A.data[:] = 1.0

        # Laplacian: L = D - A
        deg = np.array(A.sum(axis=1)).flatten()
        D = csr_matrix((deg, (range(n), range(n))), shape=(n, n))
        L = D - A

        # Normalized Laplacian for stability
        deg_inv_sqrt = np.zeros(n, dtype='float64')
        mask = deg > 0
        deg_inv_sqrt[mask] = 1.0 / np.sqrt(deg[mask])
        D_inv_sqrt = csr_matrix((deg_inv_sqrt, (range(n), range(n))), shape=(n, n))
        L_norm = D_inv_sqrt @ L @ D_inv_sqrt

        # Smallest non-trivial eigenvectors
        n_eig = min(SPECTRAL_DIM + 1, n - 1)
        eigenvalues, eigenvectors = eigsh(L_norm.astype('float64'), k=n_eig, which='SM')

        # Skip first eigenvector (constant), take next SPECTRAL_DIM
        emb = eigenvectors[:, 1:SPECTRAL_DIM+1].astype('float32')
        result = pd.DataFrame(emb, columns=[f'spectral_{i}' for i in range(emb.shape[1])])
        result['node_id'] = nodes
        SPECTRAL_CACHE[k] = result
        print(f"    Spectral embeddings: {emb.shape[1]} dims for {n} nodes")
        return result

    except Exception as e:
        print(f"    Spectral embeddings failed: {e}")
        SPECTRAL_CACHE[k] = pd.DataFrame()
        return pd.DataFrame()

print("Graph feature loader ready (v5.5: +multi-hop, +centrality, +pipe capacity).")

Graph feature loader ready (v5.5: +multi-hop, +centrality, +pipe capacity).


In [7]:
# =========================================================
# DIST TO NEAREST DRAIN (for 2D nodes)
# =========================================================
DIST_DRAIN_CACHE = {}

def load_dist_to_drain(model_id):
    if model_id in DIST_DRAIN_CACHE: return DIST_DRAIN_CACHE[model_id]
    try:
        s1d = load_static(model_id, 1)
        s2d = load_static(model_id, 2)
        x1 = fcol(s1d.columns, ('position_x',), ('position','x'), False, 'x')
        y1 = fcol(s1d.columns, ('position_y',), ('position','y'), False, 'y')
        x2 = fcol(s2d.columns, ('position_x',), ('position','x'), False, 'x')
        y2 = fcol(s2d.columns, ('position_y',), ('position','y'), False, 'y')
        if not all([x1, y1, x2, y2]):
            DIST_DRAIN_CACHE[model_id] = pd.DataFrame()
            return DIST_DRAIN_CACHE[model_id]
        from scipy.spatial import cKDTree
        tree = cKDTree(s1d[[x1, y1]].values.astype(float))
        dists, _ = tree.query(s2d[[x2, y2]].values.astype(float), k=min(3, len(s1d)))
        result = pd.DataFrame({'node_id': s2d['node_id'].values,
                               'dist_nearest_drain': dists[:, 0] if dists.ndim > 1 else dists})
        if dists.ndim > 1 and dists.shape[1] >= 2: result['dist_2nd_drain'] = dists[:, 1]
        if dists.ndim > 1 and dists.shape[1] >= 3: result['dist_3rd_drain'] = dists[:, 2]
        result['node_id'] = result['node_id'].astype('int64')
        print(f"  dist_nearest_drain Model_{model_id}: {len(result)} nodes")
        DIST_DRAIN_CACHE[model_id] = result
        return result
    except Exception as e:
        print(f"  [WARN] dist_to_drain failed: {e}")
        DIST_DRAIN_CACHE[model_id] = pd.DataFrame()
        return DIST_DRAIN_CACHE[model_id]

print("Dist-to-drain ready.")

Dist-to-drain ready.


In [8]:
# =========================================================
# RAIN FEATURES (v6: +future rain, +duration, +dry spell, +quantiles)
# =========================================================
RAIN_CACHE = {}
TIMESTEP_DURATION_CACHE = {}

def load_timestep_durations(model_id, event_dir):
    """v5.5: Load actual timestep durations from timesteps.csv."""
    k = (model_id, eid(event_dir))
    if k in TIMESTEP_DURATION_CACHE: return TIMESTEP_DURATION_CACHE[k]
    ts_path = event_dir / 'timesteps.csv'
    if not ts_path.exists():
        TIMESTEP_DURATION_CACHE[k] = None
        return None
    try:
        ts_df = read_csv(ts_path)
        # Expect columns like: timestep, datetime or time or seconds
        ts_col = fcol(ts_df.columns, ('timestep',), ('time','step'), False, 'ts')
        time_col = fcol(ts_df.columns, ('datetime','time','seconds','elapsed'),
                        ('time',), False, 'time')
        if ts_col is None or time_col is None:
            TIMESTEP_DURATION_CACHE[k] = None
            return None
        ts_df = ts_df.sort_values(ts_col).reset_index(drop=True)
        # Try to parse as seconds or datetime
        try:
            time_vals = pd.to_numeric(ts_df[time_col])
            durations = time_vals.diff().fillna(time_vals.iloc[0] if len(time_vals) > 0 else 0)
        except (ValueError, TypeError):
            try:
                time_vals = pd.to_datetime(ts_df[time_col])
                durations = time_vals.diff().dt.total_seconds().fillna(0)
            except:
                TIMESTEP_DURATION_CACHE[k] = None
                return None
        result = pd.DataFrame({
            'timestep': ts_df[ts_col].astype('int64'),
            'step_duration': durations.astype('float32'),
            'elapsed_time': time_vals.astype('float64') if pd.api.types.is_numeric_dtype(time_vals) else durations.cumsum().astype('float32'),
        })
        TIMESTEP_DURATION_CACHE[k] = result
        return result
    except Exception:
        TIMESTEP_DURATION_CACHE[k] = None
        return None

def load_rain(model_id, event_dir):
    k = (model_id, eid(event_dir))
    if k in RAIN_CACHE: return RAIN_CACHE[k]
    path_2d = dyn_path(event_dir, 2)
    df = read_csv(path_2d)
    df = ensure_nid(df)
    ts_col = fcol(df.columns, ('timestep',), ('time','step'), label='timestep')
    rain_col = fcol(df.columns, ('rainfall',), ('rain',), label='rainfall')
    rain_ts = df.groupby(ts_col)[rain_col].mean().reset_index()
    rain_ts.columns = ['timestep', 'rain_global']
    rain_ts = rain_ts.sort_values('timestep').reset_index(drop=True)
    rain = rain_ts['rain_global'].astype('float64')

    out = pd.DataFrame({
        'timestep': rain_ts['timestep'].astype('int64'),
        'rain_global': rain,
        'rain_lag1': rain.shift(1).fillna(0),
        'rain_lag2': rain.shift(2).fillna(0),
        'rain_lag3': rain.shift(3).fillna(0),
        'rain_roll3': rain.rolling(3, min_periods=1).sum(),
        'rain_roll6': rain.rolling(6, min_periods=1).sum(),
        'rain_roll12': rain.rolling(12, min_periods=1).sum(),
        'rain_roll24': rain.rolling(24, min_periods=1).sum(),
        'rain_cumsum': rain.cumsum(),
        'rain_delta': rain - rain.shift(1).fillna(0),
        'is_raining': (rain > 0).astype('int8'),
        'rain_intensity_peak': rain.expanding().max(),
    })

    for i in range(1,500):
        out[f'dodepch1k_lag_{i}'] = rain.shift(i).fillna(0)
        
    out['rain_accel'] = out['rain_delta'] - out['rain_delta'].shift(1).fillna(0)
    for alpha in [0.1, 0.05, 0.02, 0.01]:
        out[f'rain_ema_{int(1/alpha)}'] = rain.ewm(alpha=alpha, adjust=False).mean()

    total = rain.sum()
    out['rain_cum_ratio'] = (rain.cumsum() / total) if total > 0 else 0

    peak = rain.max()
    peak_ts = rain_ts.loc[rain.idxmax(), 'timestep'] if peak > 0 else 0
    max_ts = rain_ts['timestep'].max()
    out['event_total_rain'] = total
    out['event_peak_rain'] = peak
    out['time_since_rain_peak'] = out['timestep'] - peak_ts
    out['frac_event'] = out['timestep'] / (max_ts + 1)
    out['rain_phase_rising'] = (out['rain_delta'] > 0).astype('int8')
    out['rain_phase_falling'] = ((out['rain_delta'] < 0) & (rain > 0)).astype('int8')

    # v5.5: Rain duration & dry spell
    is_rain = (rain > 0).values
    rain_dur = np.zeros(len(rain), dtype='int32')
    dry_spell = np.zeros(len(rain), dtype='int32')
    cnt_rain, cnt_dry = 0, 0
    for i in range(len(rain)):
        if is_rain[i]:
            cnt_rain += 1
            cnt_dry = 0
        else:
            cnt_dry += 1
            cnt_rain = 0
        rain_dur[i] = cnt_rain
        dry_spell[i] = cnt_dry
    out['rain_duration'] = rain_dur
    out['dry_spell'] = dry_spell

    # v5.5: Intensity quantiles (expanding)
    out['rain_q75'] = rain.expanding().quantile(0.75)
    out['rain_q90'] = rain.expanding().quantile(0.90)

    # v5.5: Rain rate of change (smoothed)
    out['rain_roc_smooth'] = rain.rolling(3, min_periods=1).mean().diff().fillna(0)

    # Extra rain-shape features guided by feature importance
    eps = 1e-6
    out['rain_recent_share_6'] = out['rain_roll6'] / (out['rain_cumsum'] + eps)
    out['rain_recent_share_12'] = out['rain_roll12'] / (out['rain_cumsum'] + eps)
    out['rain_recent_share_24'] = out['rain_roll24'] / (out['rain_cumsum'] + eps)
    out['rain_roll3_over_12'] = out['rain_roll3'] / (out['rain_roll12'] + eps)
    out['rain_roll6_over_24'] = out['rain_roll6'] / (out['rain_roll24'] + eps)
    out['rain_ema_ratio_10_100'] = (out['rain_ema_10'] + eps) / (out['rain_ema_100'] + eps)
    out['rain_ema_ratio_20_100'] = (out['rain_ema_20'] + eps) / (out['rain_ema_100'] + eps)
    out['rain_ema_ratio_50_100'] = (out['rain_ema_50'] + eps) / (out['rain_ema_100'] + eps)
    out['rain_q75_gap'] = rain - out['rain_q75']
    out['rain_q90_gap'] = rain - out['rain_q90']
    out['rain_peak_gap'] = out['rain_intensity_peak'] - rain
    out['rain_peak_gap_rel'] = out['rain_peak_gap'] / (out['rain_intensity_peak'] + eps)
    out['rain_global_over_q75'] = rain / (out['rain_q75'] + eps)
    out['rain_global_over_q90'] = rain / (out['rain_q90'] + eps)
    out['rain_cumsum_per_step'] = out['rain_cumsum'] / (out['timestep'] + 1.0)
    out['dry_spell_log'] = np.log1p(out['dry_spell'])
    out['rain_duration_log'] = np.log1p(out['rain_duration'])

    # v6: Future rain features (forecast fully available — organizer confirmed)
    rain_vals = rain.values
    n_rain = len(rain_vals)
    for h in [1, 3, 5, 10]:
        shifted = rain.shift(-h).fillna(0)
        out[f'rain_future_{h}'] = shifted.astype('float32')

    # Future rolling sums: sum of rain in next w steps (excluding current)
    for w in [5, 10]:
        cs = rain.cumsum().values
        total_cs = cs[-1]
        fsum = np.zeros(n_rain, dtype='float32')
        for i in range(n_rain):
            end_idx = min(i + w, n_rain - 1)
            fsum[i] = cs[end_idx] - cs[i]
        out[f'rain_future_sum{w}'] = fsum

    # Future max: max rain in next w steps
    for w in [5, 10]:
        fmax = np.zeros(n_rain, dtype='float32')
        for i in range(n_rain):
            end_idx = min(i + w + 1, n_rain)
            if i + 1 < end_idx:
                fmax[i] = rain_vals[i+1:end_idx].max()
        out[f'rain_future_max{w}'] = fmax

    # Rain remaining after current timestep
    out['rain_remaining'] = (total - rain.cumsum()).astype('float32')

    # Rain future acceleration (will it intensify or weaken?)
    rain_future_1 = rain.shift(-1).fillna(0)
    rain_future_3 = rain.shift(-3).fillna(0)
    out['rain_future_trend'] = (rain_future_3 - rain).astype('float32')

    # v5.5: Timestep durations
    ts_dur = load_timestep_durations(model_id, event_dir)
    if ts_dur is not None:
        out = out.merge(ts_dur, on='timestep', how='left')
        if 'elapsed_time' in out.columns:
            out['rain_cumsum_per_time'] = out['rain_cumsum'] / (out['elapsed_time'].astype('float64') + 1.0)
            out['rain_roll6_per_time'] = out['rain_roll6'] / (out['elapsed_time'].astype('float64') + 1.0)

    RAIN_CACHE[k] = out
    return out

print("Rain features ready (v5.5: +duration, +dry_spell, +quantiles, +timestep_duration).")

Rain features ready (v5.5: +duration, +dry_spell, +quantiles, +timestep_duration).


In [9]:
# =========================================================
# WARMUP FEATURES + TRENDS + NEIGHBOR WARMUP
# + v5 NEW: INLET FLOW, WATER VOLUME, EDGE DYNAMICS
# =========================================================
WARMUP_CACHE = {}
WARMUP_DYN_CACHE = {}

def load_warmup(model_id, split, event_dir, node_type):
    key = (model_id, eid(event_dir), node_type, split)
    if key in WARMUP_CACHE: return WARMUP_CACHE[key]
    df = read_csv(dyn_path(event_dir, node_type))
    df = ensure_nid(df)
    ts_col = fcol(df.columns, ('timestep',), ('time','step'), label='timestep')
    wl_col = fcol(df.columns, ('water_level',), ('water','level'), label='water_level')
    warm = df[['node_id', ts_col, wl_col]].copy()
    warm.columns = ['node_id', 'timestep', 'water_level']
    warm['timestep'] = warm['timestep'].astype('int64')
    warm['water_level'] = warm['water_level'].astype('float64')
    warm = warm.sort_values(['node_id', 'timestep']).reset_index(drop=True)
    warm = warm[warm['timestep'] < WARMUP_STEPS].copy()
    warm['wi'] = warm.groupby('node_id').cumcount()
    wide = warm.pivot(index='node_id', columns='wi', values='water_level')
    wide.columns = [f'water_level_{int(c)}' for c in wide.columns]
    wide = wide.reset_index()
    for i in range(WARMUP_STEPS):
        c = f'water_level_{i}'
        if c not in wide.columns: wide[c] = np.nan
    WARMUP_CACHE[key] = wide
    return wide

def _agg_warmup_series(vals):
    clean = vals[~np.isnan(vals)]
    if len(clean) == 0:
        return np.nan, np.nan, np.nan, np.nan
    return np.mean(clean), np.max(clean), np.min(clean), clean[-1]

def load_warmup_dynamics(model_id, split, event_dir, node_type):
    key = (model_id, eid(event_dir), node_type, split)
    if key in WARMUP_DYN_CACHE: return WARMUP_DYN_CACHE[key]

    result_recs = {}

    # --- Node-level dynamic features (inlet_flow / water_volume) ---
    node_dyn = read_csv(dyn_path(event_dir, node_type))
    node_dyn = ensure_nid(node_dyn)
    ts_col = fcol(node_dyn.columns, ('timestep',), ('time','step'), label='timestep')
    node_dyn['timestep'] = node_dyn[ts_col].astype('int64')
    warmup_nodes = node_dyn[node_dyn['timestep'] < WARMUP_STEPS].copy()

    if node_type == 1:
        inlet_col = fcol(warmup_nodes.columns, ('1d_node_inlet_flow','inlet_flow'),
                         ('inlet','flow'), False, 'inlet_flow')
        if inlet_col:
            for nid, grp in warmup_nodes.groupby('node_id'):
                vals = grp[inlet_col].astype('float64').values
                mn, mx, mi, last = _agg_warmup_series(vals)
                result_recs.setdefault(int(nid), {})['inlet_flow_mean'] = mn
                result_recs.setdefault(int(nid), {})['inlet_flow_max'] = mx
                result_recs.setdefault(int(nid), {})['inlet_flow_min'] = mi
                result_recs.setdefault(int(nid), {})['inlet_flow_last'] = last
                abs_vals = np.abs(vals[~np.isnan(vals)])
                result_recs[int(nid)]['inlet_flow_abs_mean'] = np.mean(abs_vals) if len(abs_vals) else np.nan
                result_recs[int(nid)]['inlet_flow_abs_max'] = np.max(abs_vals) if len(abs_vals) else np.nan
    else:
        vol_col = fcol(warmup_nodes.columns, ('water_volume',), ('volume',), False, 'volume')
        if vol_col:
            for nid, grp in warmup_nodes.groupby('node_id'):
                vals = grp[vol_col].astype('float64').values
                mn, mx, mi, last = _agg_warmup_series(vals)
                result_recs.setdefault(int(nid), {})['wvol_mean'] = mn
                result_recs.setdefault(int(nid), {})['wvol_max'] = mx
                result_recs.setdefault(int(nid), {})['wvol_last'] = last
                clean = vals[~np.isnan(vals)]
                if len(clean) >= 2:
                    result_recs[int(nid)]['wvol_trend'] = clean[-1] - clean[0]
                else:
                    result_recs[int(nid)]['wvol_trend'] = np.nan

    del warmup_nodes, node_dyn

    # --- Edge-level dynamic features aggregated to nodes ---
    edge_path = edge_dyn_path(event_dir, node_type)
    k_ei = (model_id, node_type)
    if edge_path.exists() and k_ei in EDGE_INDEX_CACHE:
        fc_name, tc_name, ei_df = EDGE_INDEX_CACHE[k_ei]
        edge_dyn = read_csv(edge_path)
        ts_e = fcol(edge_dyn.columns, ('timestep',), ('time','step'), label='timestep')
        edge_dyn['timestep'] = edge_dyn[ts_e].astype('int64')
        warmup_edges = edge_dyn[edge_dyn['timestep'] < WARMUP_STEPS].copy()
        del edge_dyn

        if node_type == 1:
            flow_col = fcol(warmup_edges.columns, ('1d_edge_flow','edge_flow'),
                            ('edge','flow'), False, 'flow')
            vel_col = fcol(warmup_edges.columns, ('1d_edge_velocity','edge_velocity'),
                           ('velocity',), False, 'velocity')
        else:
            flow_col = fcol(warmup_edges.columns, ('2d_flow',), ('flow',), False, 'flow')
            vel_col = fcol(warmup_edges.columns, ('2d_velocity',), ('velocity',), False, 'velocity')

        if flow_col or vel_col:
            eidx_col = fcol(warmup_edges.columns, ('edge_idx','edge_id','link_id'),
                            ('edge',), False, 'edge_idx')

            edge_to_nodes = {}
            for i in range(len(ei_df)):
                edge_to_nodes[i] = (int(ei_df.iloc[i][fc_name]), int(ei_df.iloc[i][tc_name]))

            node_in_flows, node_out_flows = defaultdict(list), defaultdict(list)
            node_in_vels, node_out_vels = defaultdict(list), defaultdict(list)

            if eidx_col:
                edge_groups = warmup_edges.groupby(eidx_col)
            else:
                n_edges = len(ei_df)
                warmup_edges['_edge_idx'] = warmup_edges.groupby('timestep').cumcount()
                edge_groups = warmup_edges.groupby('_edge_idx')
                eidx_col = '_edge_idx'

            for edge_idx, grp in edge_groups:
                edge_idx = int(edge_idx)
                if edge_idx not in edge_to_nodes: continue
                from_n, to_n = edge_to_nodes[edge_idx]

                if flow_col and flow_col in grp.columns:
                    fvals = grp[flow_col].astype('float64').values
                    fmean = np.nanmean(fvals)
                    fabs_mean = np.nanmean(np.abs(fvals))
                    fabs_max = np.nanmax(np.abs(fvals))
                    node_out_flows[from_n].append((fmean, fabs_mean, fabs_max))
                    node_in_flows[to_n].append((fmean, fabs_mean, fabs_max))

                if vel_col and vel_col in grp.columns:
                    vvals = grp[vel_col].astype('float64').values
                    vmean = np.nanmean(vvals)
                    vabs_mean = np.nanmean(np.abs(vvals))
                    vabs_max = np.nanmax(np.abs(vvals))
                    node_out_vels[from_n].append((vmean, vabs_mean, vabs_max))
                    node_in_vels[to_n].append((vmean, vabs_mean, vabs_max))

            del warmup_edges

            prefix = '1d' if node_type == 1 else '2d'
            all_nids = set()
            for d in [node_in_flows, node_out_flows, node_in_vels, node_out_vels]:
                all_nids.update(d.keys())

            for nid in all_nids:
                rec = result_recs.setdefault(int(nid), {})

                if nid in node_in_flows:
                    in_f = node_in_flows[nid]
                    rec[f'{prefix}_in_flow_mean'] = np.mean([x[0] for x in in_f])
                    rec[f'{prefix}_in_flow_abs_mean'] = np.mean([x[1] for x in in_f])
                    rec[f'{prefix}_in_flow_abs_max'] = np.max([x[2] for x in in_f])

                if nid in node_out_flows:
                    out_f = node_out_flows[nid]
                    rec[f'{prefix}_out_flow_mean'] = np.mean([x[0] for x in out_f])
                    rec[f'{prefix}_out_flow_abs_mean'] = np.mean([x[1] for x in out_f])
                    rec[f'{prefix}_out_flow_abs_max'] = np.max([x[2] for x in out_f])

                if nid in node_in_vels:
                    in_v = node_in_vels[nid]
                    rec[f'{prefix}_in_vel_mean'] = np.mean([x[0] for x in in_v])
                    rec[f'{prefix}_in_vel_abs_mean'] = np.mean([x[1] for x in in_v])
                    rec[f'{prefix}_in_vel_abs_max'] = np.max([x[2] for x in in_v])

                if nid in node_out_vels:
                    out_v = node_out_vels[nid]
                    rec[f'{prefix}_out_vel_mean'] = np.mean([x[0] for x in out_v])
                    rec[f'{prefix}_out_vel_abs_mean'] = np.mean([x[1] for x in out_v])
                    rec[f'{prefix}_out_vel_abs_max'] = np.max([x[2] for x in out_v])

                if node_type == 1:
                    in_mean = rec.get(f'{prefix}_in_flow_mean', 0) or 0
                    out_mean = rec.get(f'{prefix}_out_flow_mean', 0) or 0
                    rec[f'{prefix}_net_flow'] = in_mean - out_mean

    if result_recs:
        rows = [{'node_id': nid, **feats} for nid, feats in result_recs.items()]
        result = pd.DataFrame(rows)
        result['node_id'] = result['node_id'].astype('int64')
    else:
        result = pd.DataFrame()

    WARMUP_DYN_CACHE[key] = result
    return result

def compute_warmup_trends(warmup_wide, static_df, node_type):
    wl_cols = sorted([c for c in warmup_wide.columns if c.startswith('water_level_')], key=lambda x: int(x.rsplit('_', 1)[1]))
    trend = pd.DataFrame({'node_id': warmup_wide['node_id'].astype('int64')})
    if not wl_cols:
        return trend

    vals = warmup_wide[wl_cols].values.astype('float64')
    trend['warmup_mean'] = np.nanmean(vals, axis=1)
    trend['warmup_std'] = np.nanstd(vals, axis=1)
    trend['warmup_range'] = np.nanmax(vals, axis=1) - np.nanmin(vals, axis=1)
    trend['warmup_last'] = vals[:, -1]

    recent3 = vals[:, -min(3, vals.shape[1]):]
    recent5 = vals[:, -min(5, vals.shape[1]):]
    trend['wl_recent3_mean'] = np.nanmean(recent3, axis=1)
    trend['wl_recent5_mean'] = np.nanmean(recent5, axis=1)
    trend['wl_recent3_std'] = np.nanstd(recent3, axis=1)
    trend['wl_recent5_std'] = np.nanstd(recent5, axis=1)
    trend['wl_last_vs_recent3'] = trend['warmup_last'] - trend['wl_recent3_mean']
    trend['wl_last_vs_recent5'] = trend['warmup_last'] - trend['wl_recent5_mean']
    trend['wl_peak_to_last'] = np.nanmax(vals, axis=1) - trend['warmup_last']
    trend['wl_last_to_min'] = trend['warmup_last'] - np.nanmin(vals, axis=1)

    if vals.shape[1] >= 2:
        trend['warmup_last_delta'] = vals[:, -1] - vals[:, -2]
        diffs = np.diff(vals, axis=1)
        trend['wl_diff_mean'] = np.nanmean(diffs, axis=1)
        trend['wl_diff_std'] = np.nanstd(diffs, axis=1)
        trend['wl_last3_diff_mean'] = np.nanmean(diffs[:, -min(3, diffs.shape[1]):], axis=1)
    if vals.shape[1] >= 3:
        trend['warmup_accel'] = (vals[:, -1] - vals[:, -2]) - (vals[:, -2] - vals[:, -3])
        trend['wl_recent3_slope'] = (vals[:, -1] - vals[:, -3]) / 2.0
    if vals.shape[1] >= 5:
        trend['wl_recent5_slope'] = (vals[:, -1] - vals[:, -5]) / 4.0

    x = np.arange(vals.shape[1], dtype='float64')
    xm = x.mean(); xvar = ((x - xm)**2).sum()
    if xvar > 0:
        trend['warmup_trend'] = np.nansum((x[None, :] - xm) * vals, axis=1) / xvar

    valid_mask = ~np.isnan(vals)
    vals_for_max = np.where(valid_mask, vals, -np.inf)
    vals_for_min = np.where(valid_mask, vals, np.inf)
    has_valid = valid_mask.any(axis=1)
    peak_idx = np.argmax(vals_for_max, axis=1).astype('float64')
    min_idx = np.argmin(vals_for_min, axis=1).astype('float64')
    peak_idx[~has_valid] = np.nan
    min_idx[~has_valid] = np.nan
    trend['warmup_peak_idx'] = peak_idx
    trend['warmup_min_idx'] = min_idx

    if node_type == 1:
        merge_cols = ['node_id']
        inv_c = fcol(static_df.columns, ('invert_elevation',), ('invert',), False, 'inv')
        surf_c = fcol(static_df.columns, ('surface_elevation',), ('surface',), False, 'surf')
        depth_c = fcol(static_df.columns, ('depth',), ('depth',), False, 'depth')
        area_c = fcol(static_df.columns, ('base_area',), ('base','area'), False, 'base_area')
        for c in [inv_c, surf_c, depth_c, area_c]:
            if c and c not in merge_cols:
                merge_cols.append(c)
        merged = trend.merge(static_df[merge_cols], on='node_id', how='left')

        inv_v = merged[inv_c].fillna(0).values if inv_c else np.zeros(len(merged))
        surf_v = merged[surf_c].fillna(0).values if surf_c else np.zeros(len(merged))
        pipe_d = surf_v - inv_v
        trend['fill_ratio'] = np.where(pipe_d > 0, (trend['warmup_last'].values - inv_v) / pipe_d, 0)
        trend['fill_ratio_clip'] = np.clip(trend['fill_ratio'], -2.0, 3.0)
        trend['is_surcharged'] = (trend['warmup_last'].values > surf_v).astype('int8') if surf_c else 0
        trend['head_above_invert'] = trend['warmup_last'].values - inv_v if inv_c else np.nan
        trend['headroom_to_surface'] = surf_v - trend['warmup_last'].values if surf_c else np.nan
        trend['recent3_head_above_invert'] = trend['wl_recent3_mean'].values - inv_v if inv_c else np.nan
        trend['recent3_headroom_to_surface'] = surf_v - trend['wl_recent3_mean'].values if surf_c else np.nan
        trend['warmup_peak_to_surface_gap'] = surf_v - np.nanmax(vals, axis=1) if surf_c else np.nan
        if depth_c:
            depth_v = merged[depth_c].fillna(0).values
            trend['head_above_invert_over_depth'] = np.where(depth_v > 0, trend['head_above_invert'] / depth_v, 0)
            trend['headroom_to_surface_over_depth'] = np.where(depth_v > 0, trend['headroom_to_surface'] / depth_v, 0)
            trend['depth_x_fill'] = depth_v * trend['fill_ratio']
        if area_c:
            area_v = merged[area_c].fillna(0).values
            trend['base_area_x_head'] = area_v * np.clip(trend['head_above_invert'], 0, None)
            trend['base_area_x_fill'] = area_v * trend['fill_ratio']
    return trend

def compute_neighbor_warmup(model_id, node_type, warmup_wide):
    adj = ADJ_CACHE.get((model_id, node_type), {})
    if not adj or 'water_level_9' not in warmup_wide.columns:
        return pd.DataFrame()
    wl9_map = dict(zip(warmup_wide['node_id'].astype(int),
                       warmup_wide['water_level_9'].astype('float64')))
    records = []
    for nid in warmup_wide['node_id'].astype(int):
        nid = int(nid)
        nbrs = adj.get(nid, [])
        own_wl9 = wl9_map.get(nid, np.nan)
        if nbrs:
            nbr_wl9 = [wl9_map.get(n, np.nan) for n in nbrs]
            nbr_mean = np.nanmean(nbr_wl9)
            records.append({'node_id': nid, 'nbr_mean_wl9': nbr_mean,
                           'nbr_max_wl9': np.nanmax(nbr_wl9),
                           'gradient_wl9': (own_wl9 - nbr_mean) if pd.notna(own_wl9) and pd.notna(nbr_mean) else np.nan})
        else:
            records.append({'node_id': nid, 'nbr_mean_wl9': np.nan,
                           'nbr_max_wl9': np.nan, 'gradient_wl9': np.nan})
    return pd.DataFrame(records)

print("Warmup features ready (v5: +inlet_flow, +water_volume, +edge dynamics).")

Warmup features ready (v5: +inlet_flow, +water_volume, +edge dynamics).


In [10]:
# =========================================================
# BUILD SUPERVISED FRAME (v6: +spectral embeddings, +aux targets)
# =========================================================
EVENT_FRAME_CACHE = {}

def add_importance_guided_features(df, node_type):
    eps = 1e-6

    x_col = fcol(df.columns, ('position_x',), ('position', 'x'), False, 'position_x')
    y_col = fcol(df.columns, ('position_y',), ('position', 'y'), False, 'position_y')
    if x_col and y_col:
        x = df[x_col].astype('float64')
        y = df[y_col].astype('float64')
        df['pos_radius'] = np.sqrt(x * x + y * y)
        df['pos_manhattan'] = np.abs(x) + np.abs(y)
        df['pos_angle'] = np.arctan2(y, x)
        df['pos_xy_sum'] = x + y
        df['pos_xy_diff'] = x - y

    if 'fill_ratio' in df.columns and 'rain_ema_50' in df.columns:
        df['fill_x_rain_ema_50'] = df['fill_ratio'] * df['rain_ema_50']
    if 'fill_ratio' in df.columns and 'rain_ema_100' in df.columns:
        df['fill_x_rain_ema_100'] = df['fill_ratio'] * df['rain_ema_100']
    if 'headroom_to_surface' in df.columns and 'rain_cumsum' in df.columns:
        df['headroom_x_rain_cumsum'] = df['headroom_to_surface'] * df['rain_cumsum']
    if 'head_above_invert' in df.columns and 'rain_ema_20' in df.columns:
        df['head_x_rain_ema_20'] = df['head_above_invert'] * df['rain_ema_20']
    if 'wl_recent3_mean' in df.columns and 'rain_cumsum' in df.columns:
        df['wl_recent3_x_rain_cumsum'] = df['wl_recent3_mean'] * df['rain_cumsum']
    if 'warmup_last' in df.columns and 'nbr_mean_wl9' in df.columns:
        df['warmup_last_minus_nbr_mean'] = df['warmup_last'] - df['nbr_mean_wl9']
    if 'warmup_last' in df.columns and 'nbr_max_wl9' in df.columns:
        df['warmup_last_minus_nbr_max'] = df['warmup_last'] - df['nbr_max_wl9']
    if 'water_level_9' in df.columns and 'hop2_elev_min' in df.columns:
        df['wl9_minus_hop2_elev_min'] = df['water_level_9'] - df['hop2_elev_min']
    if 'water_level_9' in df.columns and 'hop2_elev_mean' in df.columns:
        df['wl9_minus_hop2_elev_mean'] = df['water_level_9'] - df['hop2_elev_mean']
    if 'pipe_cap_total_out' in df.columns and 'pipe_cap_total_in' in df.columns:
        df['pipe_cap_net_out'] = df['pipe_cap_total_out'] - df['pipe_cap_total_in']
        df['pipe_cap_ratio_out_in'] = df['pipe_cap_total_out'] / (df['pipe_cap_total_in'] + eps)
    if 'upstream_total' in df.columns and 'pipe_cap_total_out' in df.columns:
        df['upstream_per_outcap'] = df['upstream_total'] / (df['pipe_cap_total_out'] + eps)
    if 'inlet_flow_abs_mean' in df.columns and 'pipe_cap_total_in' in df.columns:
        df['inlet_to_cap_ratio'] = df['inlet_flow_abs_mean'] / (df['pipe_cap_total_in'] + eps)
    if 'coupling_elev_diff' in df.columns and 'fill_ratio' in df.columns:
        df['coupling_x_fill'] = df['coupling_elev_diff'] * df['fill_ratio']

    return df

def build_supervised_event_frame(model_id, split, event_dir, node_type, include_target=True):
    cache_key = (model_id, eid(event_dir), node_type, split, include_target)
    if cache_key in EVENT_FRAME_CACHE:
        return EVENT_FRAME_CACHE[cache_key].copy()

    df_dyn = read_csv(dyn_path(event_dir, node_type))
    df_dyn = ensure_nid(df_dyn)
    ts_col = fcol(df_dyn.columns, ('timestep',), ('time','step'), label='timestep')
    wl_col = fcol(df_dyn.columns, ('water_level',), ('water','level'),
                  required=include_target, label='water_level')
    rain_col = fcol(df_dyn.columns, ('rainfall',), ('rain',), required=False, label='rainfall')

    keep = [ts_col, 'node_id']
    if include_target and wl_col: keep.append(wl_col)
    if node_type == 2 and rain_col: keep.append(rain_col)

    df = df_dyn[keep].copy()
    del df_dyn
    renames = {ts_col: 'timestep'}
    if include_target and wl_col: renames[wl_col] = 'water_level'
    if node_type == 2 and rain_col: renames[rain_col] = 'rain_local'
    df = df.rename(columns=renames)
    df['timestep'] = df['timestep'].astype('int64')
    df = df[df['timestep'] >= MIN_PRED_TIMESTEP].copy()

    df['model_id'] = model_id
    df['event_id'] = eid(event_dir)
    df['node_type'] = node_type

    # Static features
    static = load_static(model_id, node_type)
    df = df.merge(static, on='node_id', how='left')

    # Rain features
    rain = load_rain(model_id, event_dir)
    df = df.merge(rain, on='timestep', how='left')

    # Warmup water levels
    warmup = load_warmup(model_id, split, event_dir, node_type)
    df = df.merge(warmup, on='node_id', how='left')

    # Graph features
    gf = load_graph_feats(model_id, node_type)
    if len(gf) > 0 and 'node_id' in gf.columns:
        df = df.merge(gf, on='node_id', how='left')

    # Distance to drain (2D)
    if node_type == 2:
        dd = load_dist_to_drain(model_id)
        if len(dd) > 0 and 'node_id' in dd.columns:
            df = df.merge(dd, on='node_id', how='left')

    # Warmup trends
    trend = compute_warmup_trends(warmup, static, node_type)
    df = df.merge(trend, on='node_id', how='left')

    # Neighbor warmup
    nbr = compute_neighbor_warmup(model_id, node_type, warmup)
    if len(nbr) > 0:
        df = df.merge(nbr, on='node_id', how='left')

    # Warmup dynamics (inlet_flow, water_volume, edge flow/velocity)
    wdyn = load_warmup_dynamics(model_id, split, event_dir, node_type)
    if len(wdyn) > 0 and 'node_id' in wdyn.columns:
        df = df.merge(wdyn, on='node_id', how='left')

    if 'rain_local' not in df.columns:
        df['rain_local'] = df.get('rain_global', 0)

    # Extra handcrafted features guided by current importances
    df = add_importance_guided_features(df, node_type)

    # Interaction features (1D only)
    if 'upstream_total' in df.columns:
        df['upstream_x_rain_cumsum'] = df['upstream_total'] * df['rain_cumsum']
        df['upstream_x_rain_ema_50'] = df['upstream_total'] * df.get('rain_ema_50', 0)

    if 'inlet_flow_abs_mean' in df.columns:
        df['inlet_x_rain_cumsum'] = df['inlet_flow_abs_mean'] * df['rain_cumsum']

    if 'fill_ratio' in df.columns:
        df['fill_x_rain_cumsum'] = df['fill_ratio'] * df['rain_cumsum']

    if 'pipe_cap_total_in' in df.columns:
        df['cap_x_rain_cumsum'] = df['pipe_cap_total_in'] * df['rain_cumsum']
        df['cap_x_fill'] = df['pipe_cap_total_in'] * df.get('fill_ratio', 0)

    # v6: Spectral graph embeddings
    se = compute_spectral_embeddings(model_id, node_type)
    if len(se) > 0 and 'node_id' in se.columns:
        df = df.merge(se, on='node_id', how='left')

    # v6: Auxiliary target predictions (inlet_flow)
    if USE_AUX_TARGETS and 'AUX_MODELS' in dir() and (model_id, node_type) in AUX_MODELS:
        aux_lgb, feat_aux = AUX_MODELS[(model_id, node_type)]
        avail = [c for c in feat_aux if c in df.columns]
        if len(avail) == len(feat_aux):
            df['pred_inlet_flow'] = aux_lgb.predict(df[feat_aux]).astype('float32')
            df['pred_inlet_x_rain'] = df['pred_inlet_flow'] * df.get('rain_cumsum', 0)
        else:
            df['pred_inlet_flow'] = np.float32(0)
            df['pred_inlet_x_rain'] = np.float32(0)
    elif USE_AUX_TARGETS:
        df['pred_inlet_flow'] = np.float32(0)
        df['pred_inlet_x_rain'] = np.float32(0)

    df['node_id_cat'] = df['node_id'].astype(str)
    df['log_timestep'] = np.log1p(df['timestep'].astype('float32'))

    if include_target and USE_TARGET_DELTA and 'water_level_9' in df.columns:
        df['wl_delta_target'] = df['water_level'].astype('float64') - df['water_level_9'].astype('float64')

    for c in df.columns:
        if c == 'node_id_cat': continue
        if df[c].dtype == 'object':
            conv = pd.to_numeric(df[c], errors='coerce')
            if conv.notna().mean() >= 0.95: df[c] = conv
            else: df[c] = df[c].fillna('NA').astype(str)

    df = df.sort_values(['node_id', 'timestep']).reset_index(drop=True)
    if include_target:
        df['water_level'] = df['water_level'].astype('float64')

    # Downcast to float32 (keep target as float64)
    for c in df.columns:
        if c in (TARGET_COL, 'wl_delta_target'): continue
        if df[c].dtype == 'float64':
            df[c] = df[c].astype('float32')

    EVENT_FRAME_CACHE[cache_key] = df.copy()
    return df

def get_features(df):
    excluded = set(KEY_COLS + [TARGET_COL, 'event_id', 'wl_delta_target'])
    feat = [c for c in df.columns if c not in excluded]
    cat = [c for c in feat if df[c].dtype == 'object' or c.endswith('_cat')]
    return feat, cat

print("Frame builder ready (v6: +spectral embeddings, +future rain).")

Frame builder ready (v6: +spectral embeddings, +future rain).


In [11]:
# =========================================================
# DATASET INDEXING & SPLIT
# =========================================================
assert DATA_ROOT.exists()
model_dirs = sorted(DATA_ROOT.glob('Model_*'))
assert len(model_dirs) == 2

train_events, test_events = {}, {}
for d in model_dirs:
    m = mid_fn(d)
    train_events[m] = event_dirs(d / 'train')
    test_events[m] = event_dirs(d / 'test')
    print(f"Model {m}: {len(train_events[m])} train, {len(test_events[m])} test")

train_pool, valid_pool = {}, {}
for m in sorted(train_events):
    tr, va = train_test_split(train_events[m], test_size=VALID_EVENTS_PER_MODEL,
                               random_state=RANDOM_SEED + m, shuffle=True)
    train_pool[m] = sorted(tr, key=eid)
    valid_pool[m] = sorted(va, key=eid)
    print(f"  Split: {len(train_pool[m])} train, {len(valid_pool[m])} valid")

Model 1: 68 train, 29 test
Model 2: 69 train, 30 test
  Split: 53 train, 15 valid
  Split: 54 train, 15 valid


In [12]:
# =========================================================
# BUILD DATAFRAMES ONLY FOR (model_id=2, node_type=1)
# =========================================================
TARGET_MODEL_ID = 2
TARGET_NODE_TYPE = 1
print(f"Preparing data for Model {TARGET_MODEL_ID}, NodeType {TARGET_NODE_TYPE}")

EVENT_FRAME_CACHE.clear()
RAIN_CACHE.clear()
WARMUP_CACHE.clear()
WARMUP_DYN_CACHE.clear()
_free_memory()

train_frames = [
    build_supervised_event_frame(TARGET_MODEL_ID, 'train', ed, TARGET_NODE_TYPE, include_target=True)
    for ed in train_pool[TARGET_MODEL_ID]
]
eval_frames = [
    build_supervised_event_frame(TARGET_MODEL_ID, 'train', ed, TARGET_NODE_TYPE, include_target=True)
    for ed in valid_pool[TARGET_MODEL_ID]
]

train_df = pd.concat(train_frames, ignore_index=True)
eval_df = pd.concat(eval_frames, ignore_index=True)
del train_frames, eval_frames

EVENT_FRAME_CACHE.clear()
RAIN_CACHE.clear()
WARMUP_CACHE.clear()
WARMUP_DYN_CACHE.clear()
_free_memory()

feature_cols, cat_cols = get_features(train_df)
target_col = 'wl_delta_target' if (USE_TARGET_DELTA and 'wl_delta_target' in train_df.columns) else TARGET_COL

ordered_cols = []
for c in KEY_COLS + [TARGET_COL, 'wl_delta_target'] + feature_cols:
    if c in train_df.columns and c not in ordered_cols:
        ordered_cols.append(c)

train_df = train_df[ordered_cols].copy()
eval_df = eval_df[[c for c in ordered_cols if c in eval_df.columns]].copy()

print(f"train_df: {train_df.shape}")
print(f"eval_df : {eval_df.shape}")

# --- Build test frame for inference ---
EVENT_FRAME_CACHE.clear()
RAIN_CACHE.clear()
WARMUP_CACHE.clear()
WARMUP_DYN_CACHE.clear()
_free_memory()

test_frames = [
    build_supervised_event_frame(TARGET_MODEL_ID, 'test', ed, TARGET_NODE_TYPE, include_target=False)
    for ed in test_events[TARGET_MODEL_ID]
]
test_df = pd.concat(test_frames, ignore_index=True)
del test_frames

EVENT_FRAME_CACHE.clear()
RAIN_CACHE.clear()
WARMUP_CACHE.clear()
WARMUP_DYN_CACHE.clear()
_free_memory()

test_cols = [c for c in KEY_COLS + feature_cols if c in test_df.columns]
test_df = test_df[test_cols].copy()

print(f"test_df : {test_df.shape}")

Preparing data for Model 2, NodeType 1
  Model 2 1d edge_index: 197 edges, cols=['edge_idx', 'from_node', 'to_node']
    Edge static: 197 edges, cols=['edge_idx', 'relative_position_x', 'relative_position_y', 'length', 'diameter', 'shape', 'roughness', 'slope']
    Upstream total: max=197
    Centrality: PageRank + degree for 198 nodes
    2-hop elev features: 198 nodes
    Pipe capacity: 198 nodes
    Graph features: 198 nodes, 81 features
train_df: (2545884, 742)
eval_df : (852390, 742)
test_df : (1422036, 740)


In [13]:
dataset_info = {
    'model_id': TARGET_MODEL_ID,
    'node_type': TARGET_NODE_TYPE,
    'target_col': target_col,
    'feature_cols': feature_cols,
    'cat_cols': cat_cols,
    'n_features': len(feature_cols),
    'train_rows': len(train_df),
    'eval_rows': len(eval_df),
    'test_rows': len(test_df),
}
dataset_info

{'model_id': 2,
 'node_type': 1,
 'target_col': 'wl_delta_target',
 'feature_cols': ['timestep',
  'position_x',
  'position_y',
  'depth',
  'invert_elevation',
  'surface_elevation',
  'base_area',
  'rain_global',
  'rain_lag1',
  'rain_lag2',
  'rain_lag3',
  'rain_roll3',
  'rain_roll6',
  'rain_roll12',
  'rain_roll24',
  'rain_cumsum',
  'rain_delta',
  'is_raining',
  'rain_intensity_peak',
  'dodepch1k_lag_1',
  'dodepch1k_lag_2',
  'dodepch1k_lag_3',
  'dodepch1k_lag_4',
  'dodepch1k_lag_5',
  'dodepch1k_lag_6',
  'dodepch1k_lag_7',
  'dodepch1k_lag_8',
  'dodepch1k_lag_9',
  'dodepch1k_lag_10',
  'dodepch1k_lag_11',
  'dodepch1k_lag_12',
  'dodepch1k_lag_13',
  'dodepch1k_lag_14',
  'dodepch1k_lag_15',
  'dodepch1k_lag_16',
  'dodepch1k_lag_17',
  'dodepch1k_lag_18',
  'dodepch1k_lag_19',
  'dodepch1k_lag_20',
  'dodepch1k_lag_21',
  'dodepch1k_lag_22',
  'dodepch1k_lag_23',
  'dodepch1k_lag_24',
  'dodepch1k_lag_25',
  'dodepch1k_lag_26',
  'dodepch1k_lag_27',
  'dodepch1k_

In [14]:
SEED = 61

XGB_PARAMS = {
        'objective': 'reg:squarederror',
        'n_estimators': 50_000,
        'max_depth': 9,
        'learning_rate': 0.02,
        'eval_metric': 'rmse',
        'tree_method': 'hist',
        'device': 'gpu',
        'predictor': 'gpu_predictor',
        'random_state': 56,
}

LGB_PARAMS = {
    'objective': 'regression',
     'n_estimators': 34_000, 
    'max_depth': 8, 
    'learning_rate': 0.02,
    'reg_lambda': 3,
    'extra_trees': True,
    'subsample': 0.8, 
    'colsample_bytree': 0.8,
    'metric': 'rmse',
    'verbosity': -1,
    'n_jobs': -1,
    'random_state': 56,
    'device': 'gpu',
}

CATBOOST_PARAMS = {
    'loss_function': 'RMSE',
    'iterations': 50_000,
    'max_depth': 8,
    'learning_rate': 0.02,
    'l2_leaf_reg': 15,
    'min_data_in_leaf': 100,
    'border_count': 254,
    'eval_metric': 'RMSE',
    'task_type': 'GPU',
    'random_seed': SEED,
    'verbose': 500,
}

xgb_params = XGB_PARAMS
lgb_params = LGB_PARAMS
cbm_params = CATBOOST_PARAMS

In [15]:
feat = dataset_info['feature_cols']
feat = [x for x in feat if 'dodepch1k' not in x]
feat += ['giga_fft']
feat += ['ridge_p']
feat += [f'giga_fft_{q}' for q in [0.8, 0.9, 0.95, 0.99]]

cat = dataset_info['cat_cols']
feat_xgb = [f for f in feat if f not in cat]
target = dataset_info['target_col']
use_delta = (target == 'wl_delta_target')

In [16]:
refit_df = pd.concat([train_df, eval_df], ignore_index=True)

In [17]:
corrs = []
for i in range(1,500):
    corrs.append(refit_df[['water_level',f'dodepch1k_lag_{i}']].corr().fillna(0).values[:,1][0])

In [18]:
weights = np.array(corrs) / sum(corrs)
refit_df['giga_fft'] = 0
for i in range(1,500):
    refit_df['giga_fft'] += refit_df[f'dodepch1k_lag_{i}'].fillna(0) * weights[i-1]

test_df['giga_fft'] = 0
for i in range(1,500):
    test_df['giga_fft'] += test_df[f'dodepch1k_lag_{i}'].fillna(0) * weights[i-1]

for q in [0.8, 0.9, 0.95, 0.99]:
    refit_df[f'giga_fft_{q}'] = 0
    v = pd.Series(corrs).quantile(q)
    for i in range(1,500):
        if corrs[i-1] > v:
            refit_df[f'giga_fft_{q}'] += refit_df[f'dodepch1k_lag_{i}'].fillna(0) * corrs[i-1]

for q in [0.8, 0.9, 0.95, 0.99]:
    test_df[f'giga_fft_{q}'] = 0
    v = pd.Series(corrs).quantile(q)
    for i in range(1,500):
        if corrs[i-1] > v:
            test_df[f'giga_fft_{q}'] += test_df[f'dodepch1k_lag_{i}'].fillna(0) * corrs[i-1]

In [19]:
from sklearn.linear_model import Ridge
estim = Ridge().fit(refit_df[[f'dodepch1k_lag_{i}' for i in range(1,500)]], (refit_df[TARGET_COL]))
refit_df['ridge_p'] = estim.predict(refit_df[[f'dodepch1k_lag_{i}' for i in range(1,500)]])
test_df['ridge_p'] = estim.predict(test_df[[f'dodepch1k_lag_{i}' for i in range(1,500)]])

In [20]:
refit_df['target_lgb'] = refit_df[TARGET_COL] / refit_df['water_level_9'].values

In [21]:
lgb_valid_model = lgb.LGBMRegressor(**lgb_params)
lgb_valid_model.fit(
    refit_df[feat_xgb], refit_df['target_lgb'],
)

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


LGBMRegressor(colsample_bytree=0.8, device='gpu', extra_trees=True,
              learning_rate=0.02, max_depth=8, metric='rmse',
              n_estimators=34000, n_jobs=-1, objective='regression',
              random_state=56, reg_lambda=3, subsample=0.8, verbosity=-1)

In [22]:
xgb_valid_model = xgb.XGBRegressor(**xgb_params)
xgb_valid_model.fit(
    refit_df[feat_xgb], refit_df[target],
    verbose=300,
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device='gpu', early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.02, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=9,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=50000,
             n_jobs=None, num_parallel_tree=None, ...)

In [23]:
cb_valid_model = CatBoostRegressor(**cbm_params)
cb_valid_model.fit(
    refit_df[feat], refit_df[target],
    cat_features=cat,
    verbose=500,
)

0:	learn: 3.2964450	total: 295ms	remaining: 4h 6m 9s
500:	learn: 0.5556930	total: 18.7s	remaining: 30m 47s
1000:	learn: 0.4294900	total: 37.7s	remaining: 30m 43s
1500:	learn: 0.3695296	total: 56.4s	remaining: 30m 23s
2000:	learn: 0.3297256	total: 1m 14s	remaining: 29m 57s
2500:	learn: 0.3022409	total: 1m 33s	remaining: 29m 38s
3000:	learn: 0.2809742	total: 1m 52s	remaining: 29m 15s
3500:	learn: 0.2643997	total: 2m 10s	remaining: 28m 54s
4000:	learn: 0.2505148	total: 2m 29s	remaining: 28m 35s
4500:	learn: 0.2390459	total: 2m 47s	remaining: 28m 15s
5000:	learn: 0.2289198	total: 3m 6s	remaining: 27m 56s
5500:	learn: 0.2200053	total: 3m 24s	remaining: 27m 37s
6000:	learn: 0.2121110	total: 3m 43s	remaining: 27m 18s
6500:	learn: 0.2050783	total: 4m 1s	remaining: 26m 58s
7000:	learn: 0.1985703	total: 4m 20s	remaining: 26m 40s
7500:	learn: 0.1927271	total: 4m 39s	remaining: 26m 20s
8000:	learn: 0.1873205	total: 4m 57s	remaining: 26m 1s
8500:	learn: 0.1822162	total: 5m 15s	remaining: 25m 41s
90

In [24]:
cbm_test_pred = cb_valid_model.predict(test_df[feat])
cbm_test_pred = cbm_test_pred + test_df['water_level_9'].values

xgb_test_pred = xgb_valid_model.predict(test_df[feat_xgb])
xgb_test_pred = xgb_test_pred + test_df['water_level_9'].values

lgb_test_pred = lgb_valid_model.predict(test_df[feat_xgb])
lgb_test_pred = lgb_test_pred * test_df['water_level_9'].values

final_test_pred_ensemble = 0.4 * cbm_test_pred + 0.4 * xgb_test_pred + 0.2 * lgb_test_pred

test_pred_df = test_df[KEY_COLS].copy()
test_pred_df[TARGET_COL] = np.asarray(final_test_pred_ensemble, dtype='float32')

submission_m2_nt1_df = test_pred_df[KEY_COLS + [TARGET_COL]].copy()
submission_m2_nt1_df.to_parquet(FINAL_TEST_PRED_PARQUET_PATH, index=False)